# SIFRank & SIFRank+ — Document-Level Keyphrase Extraction

Research-oriented implementation of **SIFRank** and **SIFRank+** (Sun et al., 2020), originally prepared for a computing cluster with a dedicated conda environment.

Original experiment environment and components:
- **Python 3.6.13**
- **AllenNLP 0.8.4**
- **ELMo 2x4096_512_2048cnn_2xhighway**
- **Stanford CoreNLP 3.9.2** for POS tagging

**Reference:** Sun Y. et al., *SIFRank: A New Baseline for Unsupervised Keyphrase Extraction Based on Pre-Trained Language Model*, IEEE Access, 2020.

## Pipeline

```
Raw text
    ↓ tokenize()                — Stanford CoreNLP (spaCy fallback)
extract_candidates()            — filtered noun phrases
    ↓
build_word_freq()                — relative frequencies over the corpus
    ↓
embed_long_text()                — SIF embeddings with sliding windows (400 tokens, stride 200)
    ↓
remove_principal_component()     — remove first principal component for document + candidates
    ↓
sifrank() / sifrank+()          — candidate/document cosine similarity
                                   with an additional positional weight for SIFRank+
    ↓
Ranked keyphrases
```

## Key implementation decisions

| Step | Decision | Rationale / source |
|---|---|---|
| Embeddings | ELMo layer 1 `[0.0, 1.0, 0.0]` | Paper, §3 |
| Weighting | SIF `a / (a + p(w))`, `a = 1e-3` | Paper, §3; Arora et al. (2017) |
| Candidates | Noun phrases via Stanford CoreNLP | Paper, §3 |
| Long texts | 400-token sliding windows, stride 200 | Practical CPU/GPU memory management |
| Principal component | Remove component from document and candidate batch | Original SIFRank source implementation |
| SIFRank+ | Direct positional multiplier `1/(p1 + 3)` | Original source code rather than paper softmax |

**Environment note:** this legacy stack may be difficult to reproduce on current Python/CUDA systems. The full model weights and Stanford CoreNLP installation are external and are not included in the repository.

## 1. Environment checks

In [ ]:
import sys
import subprocess

print(f'Python version: {sys.version}')

# Verify Python version
major, minor = sys.version_info[:2]
if not (major == 3 and minor == 6):
    print(f'Python {major}.{minor}')
    print(' This notebook requires Python 3.6.')
else:
    print('Python 3.6 OK')

# Verify allennlp
try:
    import allennlp
    print(f'allennlp {allennlp.__version__} OK')
    if allennlp.__version__ != '0.8.4':
        print(f'Required version: 0.8.4, found: {allennlp.__version__}')
except ImportError:
    print('allennlp not found. pip install allennlp==0.8.4')

# Verify torch
try:
    import torch
    print(f'torch {torch.__version__} OK')
    print(f'GPU available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    print('torch not found')

# Verify Java (compatible version with Python 3.6)
try:
    result = subprocess.run(['java', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    java_ver = result.stderr.decode().split('\n')[0]
    print(f'Java: {java_ver}')
except FileNotFoundError:
    print(' Java not found. StanfordCoreNLP not available')
    print(' conda install -c conda-forge openjdk=8')

## 2. Parameters and external dependencies

Configure the locations of the external ELMo model files and Stanford CoreNLP installation here.

| Parameter | Description |
|---|---|
| `ELMO_OPTIONS_FILE` | ELMo options JSON path |
| `ELMO_WEIGHTS_FILE` | ELMo pretrained weights path |
| `STANFORD_CORENLP_PATH` | Stanford CoreNLP directory |
| `ELMO_LAYERS_WEIGHT` | Weights for three ELMo layers; paper uses `[0.0, 1.0, 0.0]` |
| `SIF_ALPHA` | SIF smoothing factor (`1e-3`) |
| `SIFRANK_PLUS_OFFSET` | SIFRank+ positional offset (`3`) |
| `USE_GPU` | Enable GPU execution when available |

The original environment used **Python 3.6.13**, **AllenNLP 0.8.4**, **PyTorch**, **NumPy**, **scikit-learn**, **pandas**, **NLTK**, **spaCy**, **Stanford CoreNLP 3.9.2**, and **Java 8**. Versions other than the historically documented ones have not been validated.

In [ ]:
import os

# Configure external model directories
ELMO_OPTIONS_FILE = 'elmo_weights/options.json'
ELMO_WEIGHTS_FILE = 'elmo_weights/weights.hdf5'
STANFORD_CORENLP_PATH = 'stanford-corenlp-full-2018-10-05'
# ELMo parameters (paper uses layer 1 (1 on 0,1,2))
# [layer0_weight, layer1_weight, layer2_weight]
ELMO_LAYERS_WEIGHT = [0.0, 1.0, 0.0]
SIF_ALPHA = 1e-3
SIFRANK_PLUS_OFFSET = 3
# Use GPU if available
USE_GPU = True   # Write True if GPU available

# Verify files
for path, name in [
    (ELMO_OPTIONS_FILE, 'ELMo options'),
    (ELMO_WEIGHTS_FILE, 'ELMo weights'),
    (STANFORD_CORENLP_PATH, 'StanfordCoreNLP')
]:
    if os.path.exists(path):
        print(f'{name}: {path}')
    else:
        print(f'{name} not found: {path}')

## 3. Loading ELMo (AllenNLP 0.8.4)

In [ ]:
import numpy as np
import torch
from typing import List, Dict, Tuple

from allennlp.modules.elmo import Elmo, batch_to_ids
from allennlp.nn.util import move_to_device

# ELMo with 3 output representations (one per layer)
# dropout = 0 for deterministic inference
elmo = Elmo(
    options_file=ELMO_OPTIONS_FILE,
    weight_file=ELMO_WEIGHTS_FILE,
    num_output_representations=3,
    dropout=0
)
elmo.eval()

if USE_GPU and torch.cuda.is_available():
    elmo = elmo.cuda()
    print('ELMo loaded on GPU')
else:
    print('ELMo loaded on CPU')


def get_token_embeddings_elmo(tokens: List[str]) -> np.ndarray:
    """
    Produces ELMo embeddings for each token.
    Combines the 3 internal layers using the weights ELMO_LAYERS_WEIGHT
    The paper uses [0.0, 1.0, 0.0] only layer 1.
    Returns
    -------
    np.ndarray of shape [N_tokens, 1024]
    """
    if not tokens:
        return np.zeros((1, 1024))

    character_ids = batch_to_ids([tokens])
    if USE_GPU and torch.cuda.is_available():
        character_ids = character_ids.to('cuda')
        #elmo = elmo.to('cuda')

    with torch.no_grad():
        embeddings = elmo(character_ids)

    # embeddings['elmo_representations'] 3 tensor list [1, T, 1024]
    emb_layers = embeddings['elmo_representations']

    # Combines the layers with the specified weights
    combined = sum(
        w * e for w, e in zip(ELMO_LAYERS_WEIGHT, emb_layers)
    )  # [1, T, 1024]

    result = combined.squeeze(0)  # [T, 1024]
    if USE_GPU:
        result = result.cpu()

    return result.numpy()


# Test
test_emb = get_token_embeddings_elmo(['tooth', 'extraction'])
print(f'Test ELMo: shape={test_emb.shape}  (expected: [2, 1024])')

## 4. POS tagging with Stanford CoreNLP

The original paper uses Stanford CoreNLP for POS tagging and noun-phrase extraction. A spaCy fallback is available if the Stanford service cannot be used.

> **Note:** Stanford CoreNLP can run into HTTP limits on very long inputs. `extract_candidates_stanford()` processes text in 5,000-character chunks to avoid this.

In [ ]:
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

STOP_WORDS = set(stopwords.words('english'))
PUNCT_CHARS = set('.,;:!?()[]{}"\'-–—/\\')
INVALID_START_POS = {'PRON', 'DET', 'CCONJ', 'SCONJ', 'PART', 'ADP', 'AUX', 'VERB'}
INVALID_START_WORDS = STOP_WORDS | {
    'this', 'these', 'those', 'that', 'such', 'each', 'both',
    'all', 'any', 'few', 'more', 'most', 'other', 'some',
    'no', 'not', 'nor', 'so', 'yet', 'either', 'neither'
}

# Try StanfordCoreNLP, fallback on spaCy
USE_STANFORD = False
try:
    from stanfordcorenlp import StanfordCoreNLP
    _stanford = StanfordCoreNLP(STANFORD_CORENLP_PATH, lang='en', quiet=True)
    USE_STANFORD = True
    print('StanfordCoreNLP loaded (as paper)')
except Exception as e:
    print(f'StanfordCoreNLP not available ({e})')
    print(' Use spaCy as fallback.')
    import spacy
    _nlp = spacy.load('en_core_web_sm')
    print('spaCy loaded as fallback')


def tokenize(text: str) -> List[str]:
    if USE_STANFORD:
        return _stanford.word_tokenize(text)
    else:
        return [t.text for t in _nlp(text) if not t.is_space]


def is_valid_candidate(phrase: str, first_pos: str = '') -> bool:
    """Filters candidates according to the paper’s criteria (Section 3)."""
    if len(phrase) <= 3:
        return False
    first_word = phrase.split()[0].lower()
    if first_word in INVALID_START_WORDS:
        return False
    if first_pos in INVALID_START_POS:
        return False
    if first_word[0].isdigit():
        return False
    if all(c in PUNCT_CHARS for c in phrase.replace(' ', '')):
        return False
    return True


def extract_candidates_stanford(text: str,
                                 min_tokens: int = 1,
                                 max_tokens: int = 5) -> List[Dict]:
    """
    Extracts noun phrases with StanfordCoreNLP by processing the text in chunks
    to avoid the CoreNLP server’s HTTP request limit
    """
    CHUNK_CHARS = 5000
    NP_POS   = {'JJ', 'JJR', 'JJS', 'NN', 'NNS', 'NNP', 'NNPS'}
    NOUN_POS = {'NN', 'NNS', 'NNP', 'NNPS'}

    seen          = {}
    global_offset = 0  # cumulative token offset across chunks

    for chunk_start in range(0, len(text), CHUNK_CHARS):
        chunk = text[chunk_start : chunk_start + CHUNK_CHARS]

        try:
            tokens   = _stanford.word_tokenize(chunk)
            pos_tags = _stanford.pos_tag(chunk)
        except Exception as e:
            print(f'   Chunk {chunk_start} failed: {e}')
            global_offset += len(tokens) if 'tokens' in dir() else 0
            continue

        total = len(tokens)

        i = 0
        while i < len(pos_tags):
            tok, pos = pos_tags[i]
            if pos in NP_POS:
                j = i
                while j < len(pos_tags) and pos_tags[j][1] in NP_POS:
                    j += 1
                if pos_tags[j-1][1] in NOUN_POS:
                    phrase_tokens = [pos_tags[k][0] for k in range(i, j)]
                    n_tok = len(phrase_tokens)
                    if min_tokens <= n_tok <= max_tokens:
                        phrase       = ' '.join(phrase_tokens)
                        phrase_clean = re.sub(r'^(the|a|an)\s+', '', phrase.lower()).strip()
                        phrase_clean = phrase_clean.strip("'.,-")
                        if phrase_clean and is_valid_candidate(phrase_clean, ''):
                            # first_occurrence in absolute tokens over the entire text
                            abs_position = global_offset + i
                            if phrase_clean not in seen:
                                seen[phrase_clean] = {
                                    'phrase':           phrase_clean,
                                    'original':         phrase,
                                    'position':         abs_position,
                                    'first_occurrence': abs_position
                                }
                i = j
            else:
                i += 1

        global_offset += total

    return list(seen.values())


def extract_candidates_spacy(text: str,
                              min_tokens: int = 1,
                              max_tokens: int = 5) -> List[Dict]:
    """Extract noun phrases with spaCy (fallback)."""
    doc   = _nlp(text)
    total = len(doc)
    seen  = {}
    for chunk in doc.noun_chunks:
        n_tok = len(chunk)
        if not (min_tokens <= n_tok <= max_tokens):
            continue
        phrase       = chunk.text.strip()
        phrase_clean = re.sub(r'^(the|a|an)\s+', '', phrase.lower()).strip()
        phrase_clean = phrase_clean.strip("'.,-")
        if not phrase_clean:
            continue
        first_pos = chunk[0].pos_
        if is_valid_candidate(phrase_clean, first_pos) and phrase_clean not in seen:
            seen[phrase_clean] = {
                'phrase':           phrase_clean,
                'original':         phrase,
                'position':         chunk.start / max(total, 1),
                'first_occurrence': chunk.start
            }
    return list(seen.values())


def extract_candidates(text: str,
                        min_tokens: int = 1,
                        max_tokens: int = 5) -> List[Dict]:
    """Dispatcher: Stanford if available, otherwise spaCy."""
    if USE_STANFORD:
        return extract_candidates_stanford(text, min_tokens, max_tokens)
    else:
        return extract_candidates_spacy(text, min_tokens, max_tokens)


# Test
test_cands = extract_candidates('Dental students experience high stress from academic and clinical demands.')
print(f'\nTest candidate extraction: {len(test_cands)} candidate')
for c in test_cands:
    print(f'   "{c["phrase"]}"  pos={c["position"]:.2f}')

## 5. SIF sentence embeddings

SIF weighting (Arora et al., 2017): `weight(w) = a / (a + p(w))`, where `a = SIF_ALPHA = 1e-3` and `p(w)` is the relative word frequency in the corpus.

Common words have high `p(w)` and therefore low weights in the embedding. Stopwords are not removed before computing SIF; they are filtered when extracting candidates.

In [ ]:
from collections import Counter
from sklearn.decomposition import TruncatedSVD


def clean_tokens_for_sif(tokens: List[str]) -> List[str]:
    """
    Removes punctuation, spaces, and very short tokens before SIF computation.
    Stopwords are NOT removed — SIF weighting handles them by assigning low weight
    (frequent words → high p(w) → low weight).
    """
    return [
        t for t in tokens
        if len(t.strip()) > 1
        and not all(c in PUNCT_CHARS for c in t.strip())
        and not t.isspace()
    ]


def build_word_freq(texts: List[str]) -> Dict[str, float]:
    """Relative word frequencies over the corpus."""
    counter = Counter()
    total   = 0
    for text in texts:
        tokens = clean_tokens_for_sif(tokenize(text))
        tokens = [t.lower() for t in tokens]
        counter.update(tokens)
        total += len(tokens)
    return {w: c / max(total, 1) for w, c in counter.items()}


def sif_weight(word: str, word_freq: Dict[str, float]) -> float:
    """SIF weight: a / (a + p(w)) with a = SIF_ALPHA."""
    p_w = word_freq.get(word.lower(), 1e-7)
    return SIF_ALPHA / (SIF_ALPHA + p_w)


def sentence_embedding_sif(tokens: List[str],
                            word_freq: Dict[str, float]) -> np.ndarray:
    """
    Computes the SIF sentence embedding:
    1. cleans the tokens
    2. computes SIF weight for each token: a / (a + p(w))
    3. computes the weighted average of ELMo embeddings
    """
    tokens_clean = clean_tokens_for_sif(tokens)
    if not tokens_clean:
        tokens_clean = tokens

    weights    = np.array([sif_weight(t, word_freq) for t in tokens_clean])
    token_embs = get_token_embeddings_elmo(tokens_clean)   # [N, 1024]

    n = min(len(weights), len(token_embs))
    weights    = weights[:n]
    token_embs = token_embs[:n]

    weighted = token_embs * weights[:, np.newaxis]         # [N, 1024]
    sent_emb = weighted.sum(axis=0) / (weights.sum() + 1e-9)
    return sent_emb   # [1024]


def remove_principal_component(embeddings: np.ndarray) -> np.ndarray:
    """
    Removes the first principal component from the set of sentence embeddings
    operates on a matrix [N_docs, 1024].
    """
    if embeddings.shape[0] < 2:
        return embeddings
    svd  = TruncatedSVD(n_components=1, random_state=42)
    svd.fit(embeddings)
    pc   = svd.components_
    proj = embeddings @ pc.T @ pc
    return embeddings - proj


print('SIF ready')

## 6. SIFRank and SIFRank+ algorithms

**Equations from the paper:**

$$\text{SIFRank}(v_{ck_i}, v_d) = \cos\text{sim}(v_{ck_i}, v_d) = \frac{v_{ck_i} \cdot v_d}{\|v_{ck_i}\| \cdot \|v_d\|}$$

$$\text{SIFRank+}(v_{ck_i}, v_d) = \frac{e^{\text{Pos}(ck_i)}}{\sum_j e^{\text{Pos}(ck_j)}} \cdot \text{SIFRank}(v_{ck_i}, v_d)$$

$$\text{Pos}(ck_i) = \frac{1}{p_1 + \mu}$$

Here, $p_1$ is the first occurrence in the document and $\mu$ is an offset.

> **SIFRank+ implementation note:** the paper presents a softmax positional weight, whereas the original reference code uses direct multiplication: `score × (1 / (p1 + offset))`. This notebook follows the reference-code variant; it is not an exact implementation of the displayed softmax equation.

> **Long documents:** `embed_long_text()` splits text into 400-token windows with stride 200. A SIF embedding is calculated per window and the results are averaged to limit memory requirements.

In [ ]:
def embed_long_text(text: str,
                    word_freq: Dict[str, float],
                    window: int = 400,
                    stride: int = 200) -> np.ndarray:
    """
    Computes the SIF embedding of a long text by splitting it into overlapping windows
    and averaging the resulting embeddings.

    ELMo does not have a fixed limit like BERT (512 tokens), but for very long texts
    (e.g., an entire corpus) GPU/CPU memory may run out
    the sliding window solves this problem and produces a more robust embedding
    because each window contributes equally.

    Parameters
    ----------
    text   : testo to embed (can be very long)
    window : number of tokens per window
    stride : step between windows (stride < window → overlapping windows)

    Returns
    -------
    np.ndarray [1024] — average embedding of the entire text
    """
    tokens = tokenize_long(text)

    if len(tokens) <= window:
        return sentence_embedding_sif(tokens, word_freq)

    window_embs = []
    for start in range(0, len(tokens) - window + 1, stride):
        window_tokens = tokens[start : start + window]
        emb = sentence_embedding_sif(window_tokens, word_freq)
        window_embs.append(emb)

    # Final window to cover the last tokens
    if (len(tokens) - window) % stride != 0:
        last_window = tokens[-window:]
        window_embs.append(sentence_embedding_sif(last_window, word_freq))

    return np.mean(window_embs, axis=0)   # [1024]

def sifrank(
    text: str,
    word_freq: Dict[str, float],
    n: int = 10,
    use_plus: bool = False
) -> List[Tuple[str, float]]:
    """
    SIFRank (use_plus=False) or SIFRank+ (use_plus=True).

    SIFRank  → score = cos_sim(candidate, document)          [eq. 17]
    SIFRank+ → score = cos_sim * (1 / (p1 + offset))         [eq. 18-19]

    The document is embedded using a sliding window (embed_long_text)
    to handle texts of any length — from a single abstract
    to an entire concatenated corpus.

    Batch-level PC removal: document + candidates together before
    cosine similarity (faithful to the original SIFRank GitHub implementation).

    When to use each:
    SIFRank → short texts (abstracts, ~100–500 words)
    SIFRank+ → long texts (full papers, entire corpora, >1000 words)
    """
    candidates = extract_candidates(text)
    if not candidates:
        return []

    # Step 1: document embedding with sliding window 
    doc_emb_raw = embed_long_text(text, word_freq)   # [1024]

    # Step 2: candidate embeddings 
    cand_embs_raw = []
    for cand in candidates:
        cand_tokens = tokenize(cand['phrase'])
        cand_emb    = sentence_embedding_sif(cand_tokens, word_freq)
        cand_embs_raw.append(cand_emb)

    # Step 3: removing PC batch-level (doc + candidate together) 
    all_embs = np.vstack([doc_emb_raw] + cand_embs_raw)  # [1+N, 1024]
    all_embs = remove_principal_component(all_embs)       # [1+N, 1024]
    doc_emb   = all_embs[0]    # [1024]
    cand_embs = all_embs[1:]   # [N, 1024]

    doc_emb_norm = doc_emb / (np.linalg.norm(doc_emb) + 1e-9)

    # Step 4: cosine similarity + scoring 
    scored = []
    for cand, cand_emb in zip(candidates, cand_embs):
        cand_norm = cand_emb / (np.linalg.norm(cand_emb) + 1e-9)
        sim = float(np.dot(doc_emb_norm, cand_norm))

        if use_plus:
            p1    = cand['first_occurrence']
            score = sim * (1.0 / (p1 + SIFRANK_PLUS_OFFSET))
        else:
            score = sim

        scored.append((cand['phrase'], round(score, 6)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:n]


print('SIFRank e SIFRank+ ready')
print('embed_long_text: sliding window 400 token, stride 200')

## 7. Corpus selection (example or local files)

Set `CORPUS_SOURCE` to `example` for the built-in small demonstration corpus, or to `files` to load your own UTF-8 text files. Each nonempty line is one document, usually formatted as `Title.. Abstract`. Multiple named domains are supported.

For `files`, edit `CORPORA` to point to existing files, such as `../data/oral_surgery_en.txt`; these research corpora are not included in the repository. Paths are relative to the working directory of the Jupyter kernel.

The example documents are illustrative, not part of the original experiments. The rest of the notebook uses the selected corpus in the same way.

In [ ]:
import pandas as pd
from pathlib import Path

# Option A: self-contained example (default), or Option B: external corpus files.
CORPUS_SOURCE = 'example'  # 'example' or 'files'
EXAMPLE_CORPORA = {
    'machine_learning': [
        'Machine learning models discover patterns in data for prediction and classification.',
        'Neural networks learn representations from training examples in deep learning systems.',
        'Natural language processing extracts keyphrases and concepts from scientific documents.'
    ],
    'oral_surgery': [
        'Oral surgery includes dental implants and reconstruction of jaw structures.',
        'Clinical studies evaluate mandibular surgery, healing, and postoperative outcomes.',
        'Dental imaging supports diagnosis and planning of maxillofacial procedures.'
    ]
}
# File-based experiments: uncomment and adapt the names and paths below.
CORPORA = {
    'oral_surgery': Path('../data/oral_surgery_en.txt'),
    'ml_junior': Path('../data/ml_en_junior.txt'),
}


def load_corpus(path: str, domain: str) -> pd.DataFrame:
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('.. ', 1)
            title = parts[0].strip() if len(parts) == 2 else ''
            abstract = parts[1].strip() if len(parts) == 2 else line
            records.append({
                'domain': domain, 'title': title, 'abstract': abstract,
                'full_text': line, 'n_words': len(line.split())
            })
    return pd.DataFrame(records)


def load_example_corpus(domain: str, documents: list) -> pd.DataFrame:
    return pd.DataFrame([
        {'domain': domain, 'title': '', 'abstract': text,
         'full_text': text, 'n_words': len(text.split())}
        for text in documents
    ])

if CORPUS_SOURCE == 'example':
    all_corpora = {name: load_example_corpus(name, docs)
                   for name, docs in EXAMPLE_CORPORA.items()}
elif CORPUS_SOURCE == 'files':
    all_corpora = {}
    for domain, path in CORPORA.items():
        if not path.is_file():
            raise FileNotFoundError(f'Corpus for {domain!r} not found: {path.resolve()}')
        all_corpora[domain] = load_corpus(str(path), domain)
else:
    raise ValueError("CORPUS_SOURCE must be 'example' or 'files'")

if not all_corpora or not any(len(df) for df in all_corpora.values()):
    raise ValueError('No documents were loaded. Check the corpus configuration.')
for domain, df in all_corpora.items():
    print(f'{domain:20s} {len(df):4d} documents '
          f'(mean {df["n_words"].mean():.0f} words)')
corpus_df = pd.concat(all_corpora.values(), ignore_index=True)
print(f'Loaded {len(corpus_df)} documents across {len(all_corpora)} domains.')

## 8. Word-frequency estimation and keyphrase extraction

In [ ]:
# Word frequency 
# built on the full corpus (all domains together)
# to obtain more robust frequency estimates
print('Building word frequency on the full corpus...')
word_freq = build_word_freq(corpus_df['full_text'].tolist())
print(f'Vocabulary: {len(word_freq):,} words')

# SIF weight check
#print('\nSIF weight check (stopwords → low weight):')
#for w in ['the', 'of', 'and', 'extraction', 'surgery', 'patient']:
#    print(f'  {w:15s}  freq={word_freq.get(w, 0):.5f}  '
#          f'SIF_weight={sif_weight(w, word_freq):.5f}')

In [ ]:
from tqdm import tqdm

full_corpus_text = ' '.join(corpus_df['full_text'].tolist())

# Stanford cannot handle very large texts in a single call
# Tokenize into chunks and concatenate the resulting tokens
def tokenize_long(text: str, chunk_size: int = 5000) -> List[str]:
    """
    Tokenizes a long text by splitting it into chunks of chunk_size characters
    (not tokens — characters, to avoid breaking words).
    each chunk is sent separately to Stanford and the tokens are concatenated.
    With spaCy this is not needed: it handles long texts natively.
    """
    if not USE_STANFORD:
        return tokenize(text)
    all_tokens = []
    for start in range(0, len(text), chunk_size):
        chunk = text[start : start + chunk_size]
        try:
            all_tokens.extend(tokenize(chunk))
        except Exception as e:
            print(f'Chunk {start}-{start+chunk_size} failed: {e}')
    return all_tokens

n_tokens_corpus = len(tokenize_long(full_corpus_text))

N_KEYPHRASES = 20
SUBSET_SIZE  = None   # None = full corpus; int for quick subset

df_run = corpus_df.head(SUBSET_SIZE).copy() if SUBSET_SIZE else corpus_df.copy()
print(f'Extraction on {len(df_run)} doc with ELMo...')

results_sifrank  = []
results_sifrankp = []

for _, row in tqdm(df_run.iterrows(), total=len(df_run)):
    text = row['full_text']
    # Automatically selects SIFRank or SIFRank+ based on text length
    # (both are computed for comparison)
    results_sifrank.append( sifrank(text, word_freq, n=N_KEYPHRASES, use_plus=False))
    results_sifrankp.append(sifrank(text, word_freq, n=N_KEYPHRASES, use_plus=True))

df_run = df_run.copy()
df_run['sifrank_keyphrases']  = results_sifrank
df_run['sifrankp_keyphrases'] = results_sifrankp
print('\nExtraction completed')

## 9. Domain-level analysis and visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Top keyphrases per domain
for domain in df_run['domain'].unique():
    df_domain = df_run[df_run['domain'] == domain]
    all_kp = [kp for doc in df_domain['sifrank_keyphrases'] for kp, _ in doc]
    top20  = Counter(all_kp).most_common(20)

    if not top20:
        continue

    labels, counts = zip(*top20)
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = cm.Blues(np.linspace(0.4, 0.9, len(labels)))
    bars = ax.barh(range(len(labels)), counts, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Frequency', fontsize=11)
    ax.set_title(f'Top 20 keyphrases — {domain} (SIFRank + ELMo)',
                 fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'sifrank_elmo_{domain}_top20.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: sifrank_elmo_{domain}_top20.png')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Top keyphrases per domain
for domain in df_run['domain'].unique():
    df_domain = df_run[df_run['domain'] == domain]
    all_kp = [kp for doc in df_domain['sifrank_keyphrases'] for kp, _ in doc]
    top_range  = Counter(all_kp).most_common(120)[99:120]

    if not top_range:
        continue

    labels, counts = zip(*top_range)
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = cm.Blues(np.linspace(0.4, 0.9, len(labels)))
    bars = ax.barh(range(len(labels)), counts, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Frequency', fontsize=11)
    ax.set_title(f'Top 100-120 keyphrases — {domain} (SIFRank + ELMo)',
                 fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'sifrank_elmo_{domain}_top100-120.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: sifrank_elmo_{domain}_top100-120.png')

### 10. Runtime considerations

ELMo inference can be computationally expensive, particularly on CPUs. Performance depends on document length, hardware, dependency versions, and the availability of compatible GPU libraries. No validated runtime benchmark is included here.

Set `USE_GPU = True` in Section 2 only when a compatible GPU and PyTorch installation are available.

## 11. Strengths and limitations

**Strengths**
- Follows the SIFRank approach (Sun et al., 2020), including ELMo, SIF weighting, principal-component removal, and Stanford CoreNLP.
- Captures semantic relevance through cosine similarity between candidate and document embeddings, rather than frequency alone.
- SIFRank+ introduces a positional preference for candidates occurring early in a document.
- Corpus-derived word frequencies allow application across specialized domains without supervised fine-tuning.
- Uses a separate document embedding for each document, preserving document-level semantics.

**Limitations**
- **Legacy environment:** AllenNLP 0.8.4 and Python 3.6 are old and may be incompatible with modern CUDA/toolchains; reproduction can require a dedicated environment.
- **Stanford CoreNLP:** depends on Java and an HTTP service, which can be fragile for very long texts.
- **ELMo computation:** inference over hundreds of documents can be costly, particularly on CPU.
- Principal-component removal is calculated on each document/candidate batch, not across documents.
- SIFRank+ here follows the original code's direct positional multiplier rather than the paper's softmax formula; scores should not be interpreted as implementing the displayed equation verbatim.